# GOLDS Training Analysis

This notebook walks through how to **analyze training results** from the GOLDS project.
We cover reading TensorBoard logs, interpreting training curves, understanding
hyperparameter sensitivity, comparing games with human-normalized scores, and
tracking self-play Elo dynamics.

**Prerequisites:** A basic understanding of reinforcement learning and PPO.
Some cells require completed training runs (TensorBoard event files or saved models).
Where real data is unavailable, we generate synthetic examples so you can still
follow along.

---

In [ ]:
# Standard imports used throughout the notebook
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Make GOLDS importable (adjust if your repo root differs)
REPO_ROOT = Path.cwd().parent  # assumes notebooks/ is one level below repo root
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# Plotting defaults
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 100})

print(f"Repo root: {REPO_ROOT}")

---
## 1. Reading TensorBoard Logs

GOLDS (and Stable-Baselines3 under the hood) writes training metrics to
**TensorBoard event files**. These binary files live under the run's log
directory, typically something like:

```
runs/<game_id>_<timestamp>/events.out.tfevents.*
```

We use [**tbparse**](https://github.com/j3soon/tbparse) to load these files
into a pandas DataFrame -- much easier than the raw TensorBoard API.

### How tbparse works

```python
from tbparse import SummaryReader
reader = SummaryReader("path/to/log_dir")
df = reader.scalars          # DataFrame with columns: step, tag, value
```

Each **tag** is a metric name (e.g. `rollout/ep_rew_mean`, `train/loss`).
Each **step** is the global timestep at which the value was recorded.

In [ ]:
# ------------------------------------------------------------------
# Load TensorBoard event files with tbparse
# ------------------------------------------------------------------
# If you have a completed training run, point LOG_DIR to its folder.
# Otherwise the try/except will generate synthetic data so the rest
# of the notebook still works.
# ------------------------------------------------------------------

LOG_DIR = REPO_ROOT / "runs"  # default SB3 log location

tb_df = None

try:
    from tbparse import SummaryReader

    # SummaryReader can point at a single run dir OR a parent dir
    # containing multiple runs.  It recursively finds event files.
    reader = SummaryReader(str(LOG_DIR))
    tb_df = reader.scalars  # pandas DataFrame

    if tb_df.empty:
        raise FileNotFoundError("No scalar data found")

    print(f"Loaded {len(tb_df)} scalar records from {LOG_DIR}")
    print(f"Available tags: {sorted(tb_df['tag'].unique())}")
    display(tb_df.head(10))

except Exception as exc:
    print(f"Could not load TensorBoard logs ({exc}).")
    print("Generating synthetic training data for demonstration...\n")

    # --- Synthetic data that mimics SB3 PPO logging ---
    np.random.seed(42)
    steps = np.arange(0, 5_000_001, 50_000)

    def _noisy_curve(final, start=0.0, noise=0.05):
        """Smooth curve from `start` to `final` with Gaussian noise."""
        t = np.linspace(0, 1, len(steps))
        base = start + (final - start) * (1 - np.exp(-4 * t))
        return base + np.random.normal(0, noise * abs(final - start), size=len(steps))

    records = []
    for step, rew, loss, ent, lr in zip(
        steps,
        _noisy_curve(final=18.0, start=-20.0, noise=0.04),
        _noisy_curve(final=0.02, start=0.15, noise=0.06),
        _noisy_curve(final=0.005, start=0.05, noise=0.04),
        np.linspace(2.5e-4, 0, len(steps)),  # linear LR decay
    ):
        records.append({"step": int(step), "tag": "rollout/ep_rew_mean", "value": rew})
        records.append({"step": int(step), "tag": "train/loss", "value": max(loss, 0.001)})
        records.append({"step": int(step), "tag": "train/entropy_loss", "value": -abs(ent)})
        records.append({"step": int(step), "tag": "train/learning_rate", "value": lr})

    tb_df = pd.DataFrame(records)
    print(f"Synthetic data: {len(tb_df)} records, tags = {sorted(tb_df['tag'].unique())}")
    display(tb_df.head(10))

### Plotting key training metrics

The most important tags logged by SB3's PPO are:

| Tag | What it tells you |
|-----|-------------------|
| `rollout/ep_rew_mean` | Average episode reward -- your primary success metric |
| `train/loss` | Total PPO loss (policy + value + entropy) |
| `train/entropy_loss` | Entropy bonus (exploration) -- should decrease slowly |
| `train/learning_rate` | Current LR (useful to verify your schedule) |

In [ ]:
# ------------------------------------------------------------------
# Plot the four key training metrics on a 2x2 grid
# ------------------------------------------------------------------

tags_to_plot = [
    ("rollout/ep_rew_mean", "Mean Episode Reward"),
    ("train/loss", "Total Loss"),
    ("train/entropy_loss", "Entropy Loss"),
    ("train/learning_rate", "Learning Rate"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, (tag, title) in zip(axes, tags_to_plot):
    subset = tb_df[tb_df["tag"] == tag].sort_values("step")
    if subset.empty:
        ax.set_title(f"{title} (no data)")
        continue
    ax.plot(subset["step"], subset["value"], linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel(tag.split("/")[-1])
    # Add a smoothed trend line
    if len(subset) > 10:
        window = max(3, len(subset) // 20)
        smoothed = subset["value"].rolling(window=window, center=True).mean()
        ax.plot(subset["step"], smoothed, color="red", linewidth=2, label="smoothed")
        ax.legend()

fig.suptitle("GOLDS Training Overview", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

---
## 2. Interpreting Training Curves

Training curves are your primary diagnostic tool. Here is what to look for:

### Healthy convergence
- **Reward** climbs steadily and levels off near the known human/expert score.
- **Loss** decreases and stabilizes (small fluctuations are normal).
- **Entropy** decays gradually -- the agent becomes more confident in its actions
  over time, but should not collapse to zero too quickly.

### Plateau
- Reward stops improving but has not reached a good score.
- Common causes: learning rate too low, not enough exploration (`ent_coef` too small),
  or the network capacity is insufficient.
- **What to try:** Increase `ent_coef`, use a cosine LR schedule, or increase `n_envs`
  to gather more diverse experience.

### Divergence
- Reward drops after initial improvement, or loss spikes upward.
- Common causes: learning rate too high, `clip_range` too large, or
  `max_grad_norm` not constraining gradient explosions.
- **What to try:** Lower `learning_rate`, tighten `clip_range` (e.g. 0.1),
  reduce `max_grad_norm`.

### Oscillation
- Reward swings wildly between high and low values.
- Common causes: too few environments (high variance in gradient estimates),
  `n_steps` too short (biased advantage estimates), or reward shaping issues.
- **What to try:** Increase `n_envs` and/or `n_steps`, smooth reward with
  `clip_reward: true`.

### Entropy collapse
- Entropy drops to near-zero early in training.
- The agent "locks in" to a narrow policy before exploring enough.
- **What to try:** Increase `ent_coef` (e.g. from 0.01 to 0.02).

In [ ]:
# ------------------------------------------------------------------
# Visualize the four training-curve patterns described above
# ------------------------------------------------------------------
np.random.seed(0)
t = np.linspace(0, 1, 200)

patterns = {
    "Convergence": 20 * (1 - np.exp(-5 * t)) + np.random.normal(0, 0.5, len(t)),
    "Plateau": 8 * (1 - np.exp(-8 * t)) + np.random.normal(0, 0.4, len(t)),
    "Divergence": 12 * (1 - np.exp(-5 * t)) - 15 * t**2 + np.random.normal(0, 0.5, len(t)),
    "Oscillation": 10 * (1 - np.exp(-3 * t)) + 4 * np.sin(15 * t) + np.random.normal(0, 0.8, len(t)),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=False)
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"]

for ax, (name, curve), color in zip(axes, patterns.items(), colors):
    ax.plot(t * 5e6, curve, linewidth=1, alpha=0.5, color=color)
    # smoothed
    smoothed = pd.Series(curve).rolling(15, center=True).mean()
    ax.plot(t * 5e6, smoothed, linewidth=2.5, color=color)
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Reward")

fig.suptitle("Common Training Curve Patterns", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Hyperparameter Sensitivity

PPO has several knobs.  Three of the most impactful for GOLDS experiments are:

| Parameter | GOLDS default | Effect |
|-----------|--------------|--------|
| `learning_rate` | 2.5e-4 | Controls step size.  Too high -> divergence.  Too low -> slow learning. |
| `ent_coef` | 0.01 | Entropy bonus weight. Higher values encourage exploration. |
| `n_envs` | 8 | Parallel environments. More envs -> lower variance gradient estimates, but more RAM. |

Below we simulate a simple sweep to illustrate how each parameter changes the
reward curve.  In a real experiment you would load separate TensorBoard runs
for each configuration.

In [ ]:
# ------------------------------------------------------------------
# Simulated hyperparameter sweep visualizations
# ------------------------------------------------------------------
np.random.seed(7)
timesteps = np.linspace(0, 10e6, 300)


def _sim_reward(final, rate, noise):
    t_norm = timesteps / timesteps.max()
    base = final * (1 - np.exp(-rate * t_norm * 10))
    return base + np.random.normal(0, noise, len(timesteps))


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Learning rate sweep ---
ax = axes[0]
for lr, final, rate, noise, style in [
    ("1e-3 (too high)", 5, 3, 3.0, "--"),
    ("2.5e-4 (default)", 18, 4, 0.8, "-"),
    ("5e-5 (too low)", 12, 1.2, 0.5, "-."),
]:
    ax.plot(timesteps, _sim_reward(final, rate, noise), label=f"lr={lr}", linestyle=style)
ax.set_title("Learning Rate", fontweight="bold")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean Reward")
ax.legend(fontsize=8)

# --- Entropy coefficient sweep ---
ax = axes[1]
for ec, final, rate, noise, style in [
    ("0.1 (too high)", 10, 2, 2.0, "--"),
    ("0.01 (default)", 18, 4, 0.8, "-"),
    ("0.001 (low)", 15, 5, 0.6, "-."),
    ("0.0 (none)", 8, 6, 0.4, ":"),
]:
    ax.plot(timesteps, _sim_reward(final, rate, noise), label=f"ent={ec}", linestyle=style)
ax.set_title("Entropy Coefficient", fontweight="bold")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean Reward")
ax.legend(fontsize=8)

# --- n_envs sweep ---
ax = axes[2]
for ne, final, rate, noise, style in [
    ("2 envs", 15, 2, 2.5, "--"),
    ("8 envs (default)", 18, 4, 0.8, "-"),
    ("16 envs", 19, 5, 0.5, "-."),
    ("32 envs", 19.5, 5.5, 0.3, ":"),
]:
    ax.plot(timesteps, _sim_reward(final, rate, noise), label=ne, linestyle=style)
ax.set_title("Number of Environments", fontweight="bold")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean Reward")
ax.legend(fontsize=8)

fig.suptitle("Hyperparameter Sensitivity (Simulated)", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### Key takeaways

1. **Learning rate** is the single most sensitive parameter. The GOLDS default
   of `2.5e-4` with a linear or cosine schedule (see `golds.training.schedules`)
   is a solid starting point.
2. **Entropy coefficient** acts as a regularizer. For simple games like Pong,
   `0.01` works well. For harder exploration problems (Montezuma's Revenge),
   you may need to increase it.
3. **n_envs** has diminishing returns -- going from 2 to 8 helps a lot,
   but 8 to 32 mostly reduces variance without dramatically changing
   the final score. More envs also use more RAM.

---
## 4. Cross-Game Comparison

Raw reward numbers are not comparable across games -- scoring 20 in Pong is
excellent, but 20 in Space Invaders is terrible.  The standard way to compare
is the **Human Normalized Score (HNS)**:

$$
\text{HNS} = \frac{\text{agent\_score} - \text{random\_score}}{\text{human\_score} - \text{random\_score}}
$$

- HNS = 0 means the agent plays no better than random.
- HNS = 1 means the agent matches human performance.
- HNS > 1 means the agent exceeds human performance.

GOLDS ships published baseline scores in `golds.results.baselines`.

In [ ]:
# ------------------------------------------------------------------
# Compute Human Normalized Scores using the GOLDS baselines module
# ------------------------------------------------------------------

try:
    from golds.results.baselines import BASELINES, human_normalized_score

    print("Available baseline games:", sorted(BASELINES.keys()))
    print()

    # Example: suppose these are your agent's mean evaluation scores.
    # Replace with real numbers from your evaluation runs!
    agent_scores = {
        "pong": 19.5,
        "breakout": 220.0,
        "space_invaders": 1100.0,
        "qbert": 11000.0,
        "seaquest": 900.0,
        "enduro": 750.0,
    }

    # Build comparison table
    rows = []
    for game_id, agent_score in sorted(agent_scores.items()):
        baseline = BASELINES.get(game_id)
        hns = human_normalized_score(game_id, agent_score)
        rows.append({
            "Game": game_id,
            "Agent Score": agent_score,
            "Random": baseline.random if baseline else None,
            "Human": baseline.human if baseline else None,
            "Published PPO": baseline.ppo if baseline else None,
            "HNS": round(hns, 3) if hns is not None else None,
        })

    comparison_df = pd.DataFrame(rows)
    display(comparison_df)

except Exception as exc:
    print(f"Could not load GOLDS baselines module: {exc}")
    print("Make sure the golds package is installed (pip install -e .)")

In [ ]:
# ------------------------------------------------------------------
# Visualize HNS across games
# ------------------------------------------------------------------

try:
    fig, ax = plt.subplots(figsize=(10, 5))

    games = comparison_df["Game"]
    hns_values = comparison_df["HNS"].astype(float)

    colors = ["#2ecc71" if v >= 1.0 else "#3498db" if v >= 0.5 else "#e74c3c" for v in hns_values]
    bars = ax.barh(games, hns_values, color=colors, edgecolor="white", linewidth=0.5)

    # Reference lines
    ax.axvline(x=1.0, color="black", linestyle="--", linewidth=1, label="Human level")
    ax.axvline(x=0.0, color="gray", linestyle=":", linewidth=1, label="Random level")

    ax.set_xlabel("Human Normalized Score (HNS)")
    ax.set_title("Cross-Game Comparison: Human Normalized Scores", fontweight="bold")
    ax.legend()

    # Annotate bars with values
    for bar, val in zip(bars, hns_values):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=9)

    fig.tight_layout()
    plt.show()

    # Median HNS is a common aggregate metric
    median_hns = hns_values.median()
    mean_hns = hns_values.mean()
    print(f"Median HNS: {median_hns:.3f}")
    print(f"Mean HNS:   {mean_hns:.3f}")

except Exception as exc:
    print(f"Skipping HNS plot: {exc}")

### Reading the HNS chart

- **Green bars** (HNS >= 1.0): Your agent matches or exceeds human performance.
- **Blue bars** (0.5 <= HNS < 1.0): Decent performance, room to improve.
- **Red bars** (HNS < 0.5): The agent is struggling. Check training curves
  for that game and consider tuning hyperparameters.

The **median HNS** across all games is the standard aggregate metric used in
papers like Mnih et al. (2015) and Schulman et al. (2017).

---
## 5. Self-Play Dynamics

For two-player games (e.g. Mortal Kombat II), GOLDS uses **self-play**: the
agent plays against past versions of itself.  Periodic snapshots of the
policy are saved and loaded as opponents.

To measure whether the agent is actually improving (and not just cycling),
we track **Elo ratings** for each snapshot. GOLDS provides
`golds.training.elo.EloTracker` for this.

### How Elo works (quick refresher)

- Every player starts at 1200.
- After a match, the winner gains points and the loser loses points.
- The amount exchanged depends on the **expected outcome**: beating a stronger
  opponent gains more points than beating a weaker one.
- The update formula: $R'_w = R_w + K \cdot (1 - E_w)$ where
  $E_w = \frac{1}{1 + 10^{(R_l - R_w)/400}}$ and $K = 32$ by default.

In [ ]:
# ------------------------------------------------------------------
# Demonstrate EloTracker with simulated self-play matches
# ------------------------------------------------------------------

try:
    from golds.training.elo import EloTracker

    # Create a tracker (no save_path = in-memory only)
    tracker = EloTracker(k_factor=32.0, initial_elo=1200.0)

    # Simulate snapshots taken every 500k timesteps during a 30M-step run
    snapshot_ids = [f"snapshot_{i * 500_000}" for i in range(1, 61)]

    # Simulate matches: later snapshots are generally stronger but not always
    np.random.seed(99)
    for i in range(1, len(snapshot_ids)):
        current = snapshot_ids[i]
        # Play against a random earlier snapshot
        opponent_idx = np.random.randint(0, i)
        opponent = snapshot_ids[opponent_idx]

        # Later snapshots win more often (probability increases with gap)
        gap = i - opponent_idx
        win_prob = min(0.95, 0.5 + 0.02 * gap)

        if np.random.random() < win_prob:
            tracker.update(winner_id=current, loser_id=opponent)
        else:
            tracker.update(winner_id=opponent, loser_id=current)

    # Show the leaderboard (top 10)
    leaderboard = tracker.get_leaderboard()
    print("Top 10 snapshots by Elo:")
    print(f"{'Rank':<6} {'Snapshot':<25} {'Elo':>8}")
    print("-" * 42)
    for rank, (snap_id, elo) in enumerate(leaderboard[:10], 1):
        print(f"{rank:<6} {snap_id:<25} {elo:>8.1f}")

except Exception as exc:
    print(f"Could not import EloTracker: {exc}")
    print("Falling back to a standalone Elo simulation...\n")

    # Minimal standalone Elo implementation for demonstration
    class SimpleElo:
        def __init__(self, k=32.0):
            self.k = k
            self.ratings = {}

        def get(self, pid):
            return self.ratings.get(pid, 1200.0)

        def update(self, winner, loser):
            rw, rl = self.get(winner), self.get(loser)
            ew = 1 / (1 + 10 ** ((rl - rw) / 400))
            self.ratings[winner] = rw + self.k * (1 - ew)
            self.ratings[loser] = rl + self.k * (0 - (1 - ew))

    tracker = SimpleElo()
    snapshot_ids = [f"snapshot_{i * 500_000}" for i in range(1, 61)]
    np.random.seed(99)
    for i in range(1, len(snapshot_ids)):
        opponent_idx = np.random.randint(0, i)
        gap = i - opponent_idx
        win_prob = min(0.95, 0.5 + 0.02 * gap)
        if np.random.random() < win_prob:
            tracker.update(snapshot_ids[i], snapshot_ids[opponent_idx])
        else:
            tracker.update(snapshot_ids[opponent_idx], snapshot_ids[i])

    leaderboard = sorted(tracker.ratings.items(), key=lambda x: x[1], reverse=True)
    print("Top 10 snapshots by Elo:")
    for rank, (snap_id, elo) in enumerate(leaderboard[:10], 1):
        print(f"  {rank}. {snap_id}: {elo:.1f}")

In [ ]:
# ------------------------------------------------------------------
# Plot Elo progression over training time
# ------------------------------------------------------------------

# Extract Elo for each snapshot in chronological order
try:
    elo_by_time = []
    for snap_id in snapshot_ids:
        # Works with both EloTracker and SimpleElo
        if hasattr(tracker, "get_rating"):
            elo = tracker.get_rating(snap_id)
        else:
            elo = tracker.get(snap_id)
        timestep = int(snap_id.split("_")[-1])
        elo_by_time.append((timestep, elo))

    elo_df = pd.DataFrame(elo_by_time, columns=["timestep", "elo"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Elo over time ---
    ax = axes[0]
    ax.plot(elo_df["timestep"], elo_df["elo"], marker="o", markersize=3, linewidth=1.5)
    ax.set_xlabel("Training Timestep")
    ax.set_ylabel("Elo Rating")
    ax.set_title("Elo Progression Over Training", fontweight="bold")
    ax.axhline(y=1200, color="gray", linestyle=":", label="Initial Elo (1200)")
    ax.legend()

    # --- Elo distribution ---
    ax = axes[1]
    ax.hist(elo_df["elo"], bins=20, edgecolor="white", color="#3498db")
    ax.axvline(x=1200, color="gray", linestyle=":", label="Initial Elo")
    ax.set_xlabel("Elo Rating")
    ax.set_ylabel("Count")
    ax.set_title("Elo Distribution Across Snapshots", fontweight="bold")
    ax.legend()

    fig.tight_layout()
    plt.show()

    # --- Win rate analysis ---
    # The latest snapshot's win rate against all others is a useful summary
    latest_elo = elo_df.iloc[-1]["elo"]
    win_rates = []
    for _, row in elo_df.iterrows():
        # Expected win probability based on Elo difference
        expected_win = 1 / (1 + 10 ** ((row["elo"] - latest_elo) / 400))
        win_rates.append(expected_win)

    elo_df["expected_win_vs_latest"] = win_rates

    print(f"\nLatest snapshot Elo: {latest_elo:.1f}")
    print(f"Expected win rate vs earliest snapshot: {elo_df.iloc[0]['expected_win_vs_latest']:.1%}")
    print(f"Expected win rate vs median snapshot:   {elo_df.iloc[len(elo_df)//2]['expected_win_vs_latest']:.1%}")

except Exception as exc:
    print(f"Skipping Elo plots: {exc}")

### What to look for in Elo curves

- **Steady upward trend**: The agent is genuinely improving over time. Each new
  snapshot is stronger than its predecessors on average.
- **Flat or cycling**: The agent may be stuck in a strategy loop -- it learns
  to beat one opponent style, then a new snapshot exploits that strategy, and
  so on. Consider using **Prioritized Fictitious Self-Play (PFSP)** to
  weight opponent sampling toward balanced matchups
  (`tracker.sample_opponent(candidates, method="pfsp")`).
- **Dips**: Occasional dips are normal, especially after the opponent pool
  refreshes. Persistent drops suggest catastrophic forgetting.

---
## Summary

| What | Tool / Module | Key Question |
|------|--------------|-------------|
| TensorBoard logs | `tbparse.SummaryReader` | Is reward increasing? Is loss stable? |
| Training curves | Visual inspection | Convergence, plateau, divergence, or oscillation? |
| Hyperparameters | Sweep + compare | Which LR / ent_coef / n_envs works best? |
| Cross-game comparison | `golds.results.baselines.human_normalized_score` | How does the agent compare to human level? |
| Self-play dynamics | `golds.training.elo.EloTracker` | Is the agent getting stronger over time? |

### Next steps

- Run `uv run golds train run --config configs/games/pong.yaml` to generate real logs.
- Re-run this notebook pointing `LOG_DIR` at the resulting run folder.
- Replace the example `agent_scores` dict with real evaluation outputs.
- For self-play games, load the Elo JSON file written during training.